# Object Recognition and Classification in Noisy Environments

This notebook builds a classical CV pipeline that segments and classifies industrial fasteners
(screws, nuts, washers) on a poorly lit scene (`details.png`).

### Why the naive detector fails
1. Aggressive morphological **closing** fills washer/nut holes and bridges nearby parts into one blob.
2. A single Hu-moment threshold (`hu[0] > 0.2`) marks some compact parts as screws.
3. An absolute area rule (`area > 1500`) never matches real washers — only merged clusters.

### Fixed pipeline
1. **Background compensation** — large-kernel morphological closing, then subtraction.
2. **Hole-preserving segmentation** — threshold + light opening; holes are filled only for connectivity.
3. **Touching-object split** — intensity watershed on oversized, non-elongated components.
4. **Feature-based classification**
   - screws: high elongation / aspect ratio
   - washers: compact rings with a significant hole ratio
   - nuts: remaining compact parts

In [ ]:
import os
import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

if not os.path.exists("details.png"):
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/13_CCL/details.png --no-check-certificate

im = cv2.imread("details.png", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(10, 6))
plt.imshow(im, cmap="gray")
plt.title("Input image (uneven illumination, shadows, specular metal)")
plt.axis("off")
plt.show()

## 1. Background removal and binary segmentation

Closing with a large elliptical kernel estimates the slowly varying background.
Subtracting it from the input turns dark fasteners into bright blobs, independent of local shading.

Important: keep a **raw** binary (holes preserved) for washer/nut cues, and a **solid** binary
(holes filled + mild close) only to reconnect thin screw shafts and run CCL.

In [ ]:
def fill_holes(binary):
    """Fill enclosed holes; return (solid_mask, hole_mask)."""
    inv = cv2.bitwise_not(binary)
    flood_mask = np.zeros((binary.shape[0] + 2, binary.shape[1] + 2), np.uint8)
    cv2.floodFill(inv, flood_mask, (0, 0), 0)
    holes = inv
    return binary | holes, holes


bg_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (55, 55))
bg = cv2.morphologyEx(im, cv2.MORPH_CLOSE, bg_kernel)
diff = cv2.GaussianBlur(cv2.subtract(bg, im), (5, 5), 0)

_, binary = cv2.threshold(diff, 26, 255, cv2.THRESH_BINARY)
binary = cv2.morphologyEx(
    binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
)
binary_raw = binary.copy()

binary_solid, hole_mask = fill_holes(binary)
binary_solid = cv2.morphologyEx(
    binary_solid, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
)

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(bg, cmap="gray")
ax[0].set_title("Estimated background")
ax[1].imshow(diff, cmap="gray")
ax[1].set_title("Background-compensated image")
ax[2].imshow(binary_raw, cmap="gray")
ax[2].set_title("Binary (holes preserved)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 2. Feature extraction, touching-object split, classification

For each connected component we compute:
- **elongation** from central moments (primary screw cue),
- **hole area / hole ratio** from the hole mask (washer vs nut),
- circularity and bounding-box aspect as supporting cues.

Large, non-elongated blobs (likely several touching parts) are split with a marker watershed
driven by local maxima of the difference image.

In [ ]:
def object_features(mask, hole_mask):
    u8 = mask.astype(np.uint8)
    area = int(u8.sum())
    moments = cv2.moments(u8)
    if moments["m00"] < 1e-6:
        return None

    hu0 = float(cv2.HuMoments(moments).flatten()[0])
    cov = np.array(
        [
            [moments["mu20"], moments["mu11"]],
            [moments["mu11"], moments["mu02"]],
        ]
    ) / moments["m00"]
    eig = np.sort(np.linalg.eigvalsh(cov))[::-1]
    elongation = float(np.sqrt(max(eig[0], 0.0) / (max(eig[1], 0.0) + 1e-12)))

    contours, _ = cv2.findContours(u8 * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        peri = cv2.arcLength(contours[0], True)
        circularity = 4 * np.pi * cv2.contourArea(contours[0]) / (peri * peri + 1e-12)
        _, _, w, h = cv2.boundingRect(contours[0])
        aspect = max(w, h) / (min(w, h) + 1e-12)
    else:
        circularity, aspect = 0.0, 1.0

    hole_area = float(((hole_mask > 0) & mask).sum())
    contours, hierarchy = cv2.findContours(u8 * 255, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    if hierarchy is not None:
        hole_area = max(
            hole_area,
            float(
                sum(
                    cv2.contourArea(contours[j])
                    for j, hinfo in enumerate(hierarchy[0])
                    if hinfo[3] != -1
                )
            ),
        )

    return {
        "area": area,
        "hu0": hu0,
        "elongation": elongation,
        "circularity": circularity,
        "aspect": aspect,
        "hole_area": hole_area,
        "hole_ratio": hole_area / (area + hole_area + 1e-12),
    }


def classify(f):
    # Screws: elongated geometry
    if f["elongation"] >= 2.0 or f["aspect"] >= 2.4:
        return "screw"
    if f["elongation"] >= 1.85 and f["hu0"] >= 0.28:
        return "screw"

    # Washers: compact rings with a meaningful central hole
    if f["elongation"] < 1.5 and f["hole_area"] >= 50 and f["hole_ratio"] >= 0.06:
        return "washer"
    if f["elongation"] < 1.35 and f["circularity"] >= 0.8 and f["hole_area"] >= 65:
        return "washer"

    return "nut"


def split_touching(mask, diff_img):
    """Split a merged blob using watershed on difference-image intensity."""
    ys, xs = np.where(mask)
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1

    roi_mask = mask[y0:y1, x0:x1]
    roi_diff = diff_img[y0:y1, x0:x1].astype(np.float32).copy()
    roi_diff[~roi_mask] = 0

    coords = peak_local_max(
        roi_diff, min_distance=10, labels=roi_mask, threshold_abs=28
    )
    if len(coords) <= 1:
        dist = cv2.distanceTransform(roi_mask.astype(np.uint8) * 255, cv2.DIST_L2, 5)
        coords = peak_local_max(
            dist, min_distance=8, labels=roi_mask, threshold_abs=3.0
        )
        field = -dist
    else:
        field = -roi_diff

    if len(coords) <= 1:
        return [mask]

    markers = np.zeros(roi_mask.shape, np.int32)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i

    labels = watershed(field, markers=markers, mask=roi_mask)
    parts = []
    for i in range(1, int(labels.max()) + 1):
        part = labels == i
        if part.sum() < 130:
            continue
        full = np.zeros_like(mask, dtype=bool)
        full[y0:y1, x0:x1] = part
        parts.append(full)
    return parts if len(parts) > 1 else [mask]


# Connected components on the solid mask
n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_solid)

objects = []
for lab in range(1, n_labels):
    area = int(stats[lab, cv2.CC_STAT_AREA])
    if area < 150:
        continue

    mask = labels == lab
    f = object_features(mask, hole_mask)
    touching = f["elongation"] < 1.9 and (
        area > 1300 or (area > 950 and f["circularity"] < 0.5)
    )
    if touching:
        objects.extend(split_touching(mask, diff))
    else:
        objects.append(mask)

# Prefer the hole-preserving silhouette; fall back if raw mask is too broken
clean_objects = []
for mask in objects:
    clipped = mask & (binary_raw > 0)
    if clipped.sum() < 0.55 * mask.sum():
        clipped = mask & ~(hole_mask > 0)
    if clipped.sum() >= 130:
        clean_objects.append(clipped)

screws = np.zeros_like(im)
nuts = np.zeros_like(im)
washers = np.zeros_like(im)
counts = {"screw": 0, "nut": 0, "washer": 0}

print(f"{'id':>3} {'area':>5} {'elong':>6} {'circ':>5} {'hole':>5} {'hRatio':>6}  class")
for i, mask in enumerate(clean_objects, 1):
    f = object_features(mask, hole_mask)
    cls = classify(f)

    # Drop tiny watershed shards that are not real fasteners
    if cls == "screw" and f["area"] < 450:
        continue
    if cls == "nut" and f["area"] < 200 and f["hole_area"] < 10 and f["circularity"] < 0.5:
        continue

    counts[cls] += 1
    if cls == "screw":
        screws[mask] = 255
    elif cls == "washer":
        washers[mask] = 255
    else:
        nuts[mask] = 255

    print(
        f"{i:3d} {f['area']:5d} {f['elongation']:6.2f} {f['circularity']:5.2f} "
        f"{f['hole_area']:5.0f} {f['hole_ratio']:6.3f}  {cls}"
    )

print("\nDetected counts:", counts)

## 3. Classification result

Red = screws, green = nuts, blue = washers.

In [ ]:
overlay = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB).astype(np.float32)
overlay[screws > 0] = overlay[screws > 0] * 0.35 + np.array([255, 50, 50]) * 0.65
overlay[nuts > 0] = overlay[nuts > 0] * 0.35 + np.array([50, 210, 50]) * 0.65
overlay[washers > 0] = overlay[washers > 0] * 0.35 + np.array([50, 90, 255]) * 0.65

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
ax[0, 0].imshow(im, cmap="gray")
ax[0, 0].set_title("Input")
ax[0, 1].imshow(binary_raw, cmap="gray")
ax[0, 1].set_title("Segmentation (holes preserved)")
ax[1, 0].imshow(np.dstack([screws, nuts, washers]))
ax[1, 0].set_title("R = screws,  G = nuts,  B = washers")
ax[1, 1].imshow(overlay.astype(np.uint8))
ax[1, 1].set_title(f"Overlay  {counts}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(screws, cmap="gray")
ax[0].set_title(f"Screws ({counts['screw']}) — high elongation")
ax[1].imshow(nuts, cmap="gray")
ax[1].set_title(f"Nuts ({counts['nut']}) — compact, small hole")
ax[2].imshow(washers, cmap="gray")
ax[2].set_title(f"Washers ({counts['washer']}) — ring / large hole ratio")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

### Workspace cleanup

In [ ]:
for path in glob.glob("details.png"):
    try:
        os.remove(path)
    except OSError:
        pass

print("Temporary resources removed from the local filesystem.")